# 第35章 美化、注释与导出

通过有限配色、刻度格式、重点注释和规范导出提升图表可读性。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

图表进入报告、汇报或作品集前的统一整理阶段。

## 数据结构

任何已经确定分析结论的Matplotlib图表。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 修改 annotate 的 xytext 参数（如 (-40, 35) 或 (-70, 20)），调整注释箭头位置
2. 将 savefig 的 dpi 从 180 改为 300，对比不同分辨率的导出效果
3. 修改 grid 的 alpha 参数（如 0.05 或 0.3），说明网格透明度对可读性的影响


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from js import window
base_url = window.location.origin
transactions = pd.read_csv(f"{base_url}/datasets/uci_online_retail_200k.csv", parse_dates=["InvoiceDate"])
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]
transactions["month"] = transactions["InvoiceDate"].dt.to_period("M").astype("string")
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
monthly_summary = completed.groupby("month").agg(sales=("amount", "sum"), orders=("InvoiceNo", "nunique"))
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
profit = sales * 0.18
top_countries = completed.groupby("Country")["amount"].sum().nlargest(4).index
country_rows = transactions[transactions["Country"].isin(top_countries)].copy()
country_rows["flow"] = np.where(country_rows["Quantity"] > 0, "销售", "退货")
country_rows["amount_abs"] = country_rows["amount"].abs()
regional_summary = country_rows.pivot_table(index="Country", columns="flow", values="amount_abs", aggfunc="sum", fill_value=0) / 10_000
regions = regional_summary.index.to_numpy()
online = regional_summary.get("销售", pd.Series(0, index=regional_summary.index)).to_numpy()
offline = regional_summary.get("退货", pd.Series(0, index=regional_summary.index)).to_numpy()
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"UCI Online Retail：{len(transactions):,} 行；图表使用聚合结果与固定样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
from matplotlib.ticker import FuncFormatter

fig, ax = plt.subplots(figsize=(8, 4.2))
colors = ["#9aa0a6"] * 5 + ["#1a73e8"]
bars = ax.bar(months, sales, color=colors)
ax.bar_label(bars, padding=4, fmt="%.0f")
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.0f}万"))
ax.set(title="6月销售额达到半年最高", xlabel="月份", ylabel="销售额")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="y", alpha=0.15)
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
from io import BytesIO

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, profit, marker="o", color="#188038", linewidth=2)
peak = int(profit.argmax())
ax.annotate(f"最高 {profit[peak]} 万元", (months[peak], profit[peak]), xytext=(-60, 28), textcoords="offset points", arrowprops={"arrowstyle": "->", "color": "#188038"})
ax.set(title="利润在6月达到最高", ylabel="利润（万元）")
fig.tight_layout()
buffer = BytesIO()
fig.savefig(buffer, format="png", dpi=180, bbox_inches="tight")
print(f"导出PNG大小: {buffer.getbuffer().nbytes / 1024:.1f} KB")
plt.show()


## 3. 参数说明

- tick formatter：刻度格式
- annotate：注释
- spines：边框
- savefig：导出


## 4. 结果解读

视觉重点应与结论一致；标题表达结论，坐标轴表达指标和单位。


## 常见误区

- 装饰多于信息
- 颜色数量过多
- 数据标签互相遮挡
- 导出时标题被裁切


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
sales_growth = np.r_[np.nan, np.diff(sales) / sales[:-1]]
ax.plot(months, sales_growth * 100, marker="o", color="#1a73e8")
ax.axhline(0, color="#9aa0a6", linewidth=1)
ax.set(title="除3月外，月度销售额保持增长", ylabel="环比增长率（%）")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.15)
fig.tight_layout()
plt.show()


## 本章小结

通过有限配色、刻度格式、重点注释和规范导出提升图表可读性。


### 你已经掌握

- 判断美化、注释与导出的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 图表进入报告、汇报或作品集前的统一整理阶段。 |
| 数据结构 | 任何已经确定分析结论的Matplotlib图表。 |
| 结果解读 | 视觉重点应与结论一致；标题表达结论，坐标轴表达指标和单位。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `tick formatter` | 刻度格式 |
| `annotate` | 注释 |
| `spines` | 边框 |
| `savefig` | 导出 |


### 需要注意

- 装饰多于信息
- 颜色数量过多
- 数据标签互相遮挡
- 导出时标题被裁切


### 完成检查

- [ ] 能判断什么问题适合使用美化、注释与导出
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
